In [3]:
"""
Build monthly Argo profile locations (sampling-probability maps) on a NEMO model grid.

Generalised so the user can specify:
  1. A time period (start_date / end_date) to construct monthly counts
  2. An optional subregion of the ocean model grid (lat/lon bounding
     box) to restrict the analysis to.

Usage: edit the CONFIG block below, then run the script.
"""

import glob
import numpy as np
import pandas as pd
import xarray as xr
from sklearn.neighbors import BallTree

# ─────────────────────────────────────────────────────────────────────────
# CONFIG — edit these to control time period / subregion / paths
# ─────────────────────────────────────────────────────────────────────────

# JASMIN Object Store domain URL for NEMO eORCA1 JRA55v1 NPD data
domain_URL = "https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/domain_cfg"
#domain_URL ="https://noc-msm-o.s3-ext.jc.rl.ac.uk/noc-npd-era5/npd-eorca12-era5v1"

# Glob pattern for input parquet profile files
PROFILE_GLOB = (
    "/dssgfs01/working/shapat/OSSE/OceanOSSE/OceanOSSE/sampling/argo/"
    "data/profiles_g10/EN.4.2.2.f.profiles.g10.2001_2026.parquet"
)

# Output directory for the resulting NetCDF
OUTPUT_DIR = "/dssgfs01/working/shapat/OSSE/OceanOSSE/OceanOSSE/sampling/argo/data/profiles_g10"

# Column in the parquet data holding the profile date/time
DATE_COL = "JULD"  # change if needed, e.g. "TIME"

# --- Time period -----------------------------------------------------------
# Set to None to use the full range of dates present in the data.
# Otherwise use "YYYY-MM-DD" strings (inclusive of both ends).
START_DATE = None  # e.g. "2010-01-01"
END_DATE = None    # e.g. "2015-12-31"

# --- Subregion ---------------------------------------------------------
# Set REGION to None to use the full model grid.
# Otherwise provide a dict with lat/lon bounding box (degrees).
# Longitudes should be in the same convention as glamt (e.g. for NEMO its -180..180
# or 0..360 — check ds_domain["glamt"] if unsure).
REGION = None
#REGION = {"lat_min": 10, "lat_max": 50, "lon_min": -30, "lon_max": 10}

# Example: REGION = {"lat_min": -10, "lat_max": 10, "lon_min": -30, "lon_max": 20}

# ─────────────────────────────────────────────────────────────────────────
# 1. Load model grid (domain_cfg), build a WET-POINT-ONLY land-sea mask,
#    and optionally subset to REGION
# ─────────────────────────────────────────────────────────────────────────

#ds_domain = xr.open_zarr(f"{domain_URL}", consolidated=True, chunks={})
#ds_domain = xr.open_dataset("/dssgfs01/working/shapat/OSSE/OceanOSSE/OceanOSSE/sampling/argo/data/profiles_g10/mesh_mask.nc")
ds_domain = xr.open_dataset("/dssgfs01/scratch/npd/simulations/Domains/eORCA025/domain_cfg.nc")
lon_model = ds_domain["glamt"][0,:,:].values
lat_model = ds_domain["gphit"][0,:,:].values

ny, nx = lat_model.shape

# --- Build a surface T-point wet mask (True = ocean, False = land) --------
# Different domain_cfg / mesh_mask files expose this differently, so try
# the common variants in order of preference.
if "top_level" in ds_domain:
    # top_level == 0 at a T-point means it is permanently land
    wet_mask = (ds_domain["top_level"].values > 0)
elif "bottom_level" in ds_domain:
    wet_mask = (ds_domain["bottom_level"].values > 0)
elif "tmask" in ds_domain:
    tmask = ds_domain["tmask"].values
    # tmask is (t, z, y, x) ; take surface level
    if tmask.ndim == 4:
        wet_mask = tmask[0, 0] > 0
    elif tmask.ndim == 3:
        wet_mask = tmask[0] > 0
    else:
        wet_mask = tmask > 0
else:
    raise KeyError(
        "Could not find a land-sea mask variable in domain_cfg "
        "(looked for 'top_level', 'bottom_level', 'tmask'). Inspect "
        "ds_domain.data_vars and set wet_mask manually."
    )

wet_mask = wet_mask.astype(bool)
if not wet_mask.any():
    raise ValueError("Wet mask is empty — check the mask variable/logic above.")

print(f"Wet T-points: {wet_mask.sum()} / {wet_mask.size} grid cells")

# --- Apply optional subregion on top of the wet mask -----------------------
if REGION is not None:
    region_box = (
        (lat_model >= REGION["lat_min"])
        & (lat_model <= REGION["lat_max"])
        & (lon_model >= REGION["lon_min"])
        & (lon_model <= REGION["lon_max"])
    )
    if not region_box.any():
        raise ValueError(
            "REGION bounding box does not overlap the model grid — check "
            "lat/lon ranges and longitude convention (e.g. -180..180 vs 0..360)."
        )
    index_mask = wet_mask & region_box
    
    if not index_mask.any():
        raise ValueError(
            "REGION bounding box contains no wet ocean points — check the "
            "bounding box, or it may fall entirely on land."
        )
else:
    index_mask = wet_mask

# --- After building index_mask, get the tight 2D bounding slice ---
if REGION is not None:
    rows_with_data = np.where(index_mask.any(axis=1))[0]
    cols_with_data = np.where(index_mask.any(axis=0))[0]
    row_min, row_max = rows_with_data[0], rows_with_data[-1] + 1
    col_min, col_max = cols_with_data[0], cols_with_data[-1] + 1
else:
    row_min, col_min = 0, 0
    row_max, col_max = ny, nx

# Cropped grid metadata (used when saving NetCDF)
lat_save = lat_model[row_min:row_max, col_min:col_max]
lon_save = lon_model[row_min:row_max, col_min:col_max]
ny_save  = row_max - row_min
nx_save  = col_max - col_min

# Flat indices of cells eligible for nearest-neighbour matching: wet,
# and inside REGION if one was given. Land points are NEVER included,
# so a query can never return a land cell as "nearest".
index_flat_idx = np.flatnonzero(index_mask.ravel())
grid_points = np.column_stack(
    [lat_model.ravel()[index_flat_idx], lon_model.ravel()[index_flat_idx]]
)

# GeoBallTree: haversine-distance BallTree over wet ocean points only.
# wet-only point set so land cells can never be returned as "nearest".)
tree = BallTree(np.deg2rad(grid_points), metric="haversine")

# ─────────────────────────────────────────────────────────────────────────
# 2. Load Argo profile data
# ─────────────────────────────────────────────────────────────────────────

files = sorted(glob.glob(PROFILE_GLOB))
if not files:
    raise FileNotFoundError(f"No parquet files matched: {PROFILE_GLOB}")

df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

# ─────────────────────────────────────────────────────────────────────────
# 3. Parse date, filter to requested time period, extract MONTH
# ─────────────────────────────────────────────────────────────────────────

df[DATE_COL] = pd.to_datetime(df[DATE_COL])

if START_DATE is not None:
    df = df[df[DATE_COL] >= pd.Timestamp(START_DATE)]
if END_DATE is not None:
    df = df[df[DATE_COL] <= pd.Timestamp(END_DATE)]

if len(df) == 0:
    raise ValueError(
        "No profiles remain after applying START_DATE/END_DATE filter — "
        "check the requested time period against the data's date range."
    )

df["MONTH"] = df[DATE_COL].dt.month

print("Profiles after time filtering:", len(df))
print(df["MONTH"].value_counts().sort_index())

# ─────────────────────────────────────────────────────────────────────────
# 4. Match each profile to its nearest model grid cell within the region
# ─────────────────────────────────────────────────────────────────────────

obs_points = np.column_stack([df["LATITUDE"], df["LONGITUDE"]])
dist, ind = tree.query(np.deg2rad(obs_points), k=1)

# `ind` indexes into grid_points (i.e. into index_flat_idx — wet points,
# optionally restricted to REGION), so map back to the original flat grid
# index, then to 2D (iy, ix). Because land was excluded when building the
# tree, this can never resolve to a land cell.
flat_idx_full = index_flat_idx[ind[:, 0]]
iy, ix = np.unravel_index(flat_idx_full, lat_model.shape)

# Optional sanity check / diagnostic: how far (km) was each profile from
# its matched wet cell? Useful for spotting coastal profiles assigned to
# a wet cell far away because of a coarse grid.
EARTH_RADIUS_KM = 6371.0
match_dist_km = dist[:, 0] * EARTH_RADIUS_KM
print(
    f"Nearest-wet-cell distance (km): "
    f"mean={match_dist_km.mean():.1f}, max={match_dist_km.max():.1f}"
)

df["iy"] = iy
df["ix"] = ix

# ─────────────────────────────────────────────────────────────────────────
# 5. Build monthly counts (full grid shape; cells outside REGION stay zero)
# ─────────────────────────────────────────────────────────────────────────

counts = np.zeros((12, ny_save, nx_save), dtype=np.float32)
for m in range(1, 13):
    subset = df[df["MONTH"] == m]
    if len(subset) == 0:
        continue
     # Shift indices into the cropped coordinate frame
    iy_crop = subset["iy"].values - row_min
    ix_crop = subset["ix"].values - col_min
    np.add.at(counts[m - 1], (iy_crop, ix_crop), 1)
    #np.add.at(counts[m - 1], (subset["iy"].values, subset["ix"].values), 1)

print("Total profiles mapped:", counts.sum())
print("Counts per month:", counts.sum(axis=(1, 2)))

# ─────────────────────────────────────────────────────────────────────────
# 6. Convert counts to probabilities (normalised within region, per month)
# ─────────────────────────────────────────────────────────────────────────

month_totals = counts.sum(axis=(1, 2), keepdims=True)
month_totals = np.where(month_totals == 0, 1, month_totals)
prob = counts / month_totals

# ─────────────────────────────────────────────────────────────────────────
# 7. Save as NetCDF, with metadata describing the time period / region used
# ─────────────────────────────────────────────────────────────────────────

ds_prob = xr.Dataset(
    {"argo_probability": (["month", "y", "x"], prob)},
    coords={
        "month": np.arange(1, 13),
        "lat_model": (["y", "x"], lat_save),
        "lon_model": (["y", "x"], lon_save),
    },
)

actual_start = df[DATE_COL].min().strftime("%Y-%m-%d")
actual_end = df[DATE_COL].max().strftime("%Y-%m-%d")

ds_prob["argo_probability"].attrs = {
    "long_name": "Monthly Argo sampling probability",
    "units": "1",
}
ds_prob.attrs["time_period"] = f"{actual_start} to {actual_end}"
ds_prob.attrs["region"] = str(REGION) if REGION is not None else "full model grid"
ds_prob.attrs["matching"] = (
    "Profiles matched to nearest WET T-point only (land-sea masked "
    "GeoBallTree, haversine distance); land cells cannot be assigned."
)

# Build an informative, period/region-aware filename
period_tag = f"{actual_start[:4]}_{actual_end[:4]}"
region_tag = "subregion" if REGION is not None else "full"
out_name = f"argo_probability_NEMO025_grid_{period_tag}_{region_tag}.nc"
out_path = f"{OUTPUT_DIR}/{out_name}"

ds_prob.to_netcdf(out_path)
print(f"Saved: {out_path}")

Wet T-points: 910212 / 1736640 grid cells
Profiles after time filtering: 2671012
MONTH
1     221213
2     204123
3     223896
4     216594
5     225270
6     218645
7     224883
8     225885
9     222770
10    230173
11    223723
12    233837
Name: count, dtype: int64
Nearest-wet-cell distance (km): mean=8.7, max=37.3
Total profiles mapped: 2.671012e+06
Counts per month: [221213. 204123. 223896. 216594. 225270. 218645. 224883. 225885. 222770.
 230173. 223723. 233837.]
Saved: /dssgfs01/working/shapat/OSSE/OceanOSSE/OceanOSSE/sampling/argo/data/profiles_g10/argo_probability_NEMO025_grid_2001_2025_full.nc
